## Libraries set-up

In [1]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.express as px

## A) Dataset build-up

In [2]:
# 1. Configuration: TICKERS / Sector mapping
sector_map = {
    'Solar': ['FSLR', 'ENPH', 'SEDG', 'NXT', 'CSIQ', 'RUN', 'SHLS', 'DQ', 'JKS', 'SPWR', 'MAXN'],
    'Wind': ['GEV', 'VWS.CO', 'BEP', 'CWEN', 'ORA', 'TPIC', 'ORSTED.CO'],
    'Nuclear': ['CEG', 'CCJ', 'UUUU', 'LEU', 'UEC', 'SMR', 'OKLO', 'BWXT', 'FLR', 'NXE', 'DNN','IMSR'],
    'Hydrogen': ['BE', 'PLUG', 'NEL.OL', 'ITM.L', 'HYSR', 'FCEL', 'BLDP'],
    'Storage & Grid': ['TSLA', 'FLNC', 'STEM', 'EOSE', 'GWH', 'NRGV', 'GNRC', 'AYI', 'HUBB', 'PWR', 'VRT', 'EAT'],
    'Integrated Energy': ['NEE', 'XOM', 'CVX', 'SHEL', 'TTE', 'BP', 'EQNR', 'IBE.MC', 'ENEL.MI']
}

all_tickers = [ticker for sublist in sector_map.values() for ticker in sublist]
ticker_to_sector = {ticker: sector for sector, tickers in sector_map.items() for ticker in tickers}

# 2. Fetch Fresh Data (Summary and Flipped Historical)
def fetch_data():
    summary_data, historical_prices = [], pd.DataFrame()
    end_date = datetime.now()
    start_date = end_date - timedelta(days=2*365)

    print(f"Fetching data for {len(all_tickers)} companies...")
    for ticker in all_tickers:
        try:
            stock = yf.Ticker(ticker)
            info, hist = stock.info, stock.history(start=start_date, end=end_date)['Close']
            if hist.empty: continue
            hist.index = hist.index.tz_localize(None).date

            summary_data.append({
                'Ticker': ticker, 'Name': info.get('longName', ticker), 'Sector': ticker_to_sector.get(ticker),
                'Market Capitalization': info.get('marketCap'), 'Trading Volume (Avg)': info.get('averageVolume'),
                '52 Week High': info.get('fiftyTwoWeekHigh'), '52 Week Low': info.get('fiftyTwoWeekLow'),
                'Last Price': info.get('currentPrice') or info.get('regularMarketPreviousClose')
            })
            historical_prices[ticker] = hist
        except: continue
    return pd.DataFrame(summary_data), historical_prices

df_summary, df_history_wide = fetch_data()




Fetching data for 58 companies...


In [3]:
# 3. FLIP THE AXIS & ENRICH DATA
# Melt Wide to Long
df_history_long = df_history_wide.reset_index().melt(id_vars='index', var_name='Ticker', value_name='Price')
df_history_long.rename(columns={'index': 'Date'}, inplace=True)

# Calculate Price Change regarding previous close (per Ticker)
df_history_long.dropna(subset=['Price'], inplace=True)
df_history_long = df_history_long.sort_values(['Ticker', 'Date'])
df_history_long['Price Change'] = df_history_long.groupby('Ticker')['Price'].pct_change(fill_method=None)

# ADDED: Calculate Cumulative Change (Return since the start of the 2-year period)
df_history_long['Cumulative Change'] = df_history_long.groupby('Ticker')['Price'].transform(lambda x: (x / x.iloc[0]) - 1)

# Sort summary to identify Top 5
df_summary = df_summary.sort_values(by='Market Capitalization', ascending=False)
df_top_5 = df_summary.head(5)
top_5_tickers = df_top_5['Ticker'].tolist()

# NEW: Merge Name and Create 'Top Five?' column
df = df_history_long.merge(df_summary[['Ticker', 'Name', 'Sector']], on='Ticker', how='left') # Added 'Sector' here
df['Top Five?'] = df['Ticker'].apply(lambda x: 'Yes' if x in top_5_tickers else 'No')

## B) Plotting Cumulative Change

In [4]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pandas')

#1 Timeframe set-up
min_date, max_date = df['Date'].min(), df['Date'].max()
first_month_label = min_date.strftime("%b %Y")
last_month_label = max_date.strftime("%b %Y")

# 2. Data Aggregation for Sectors
df_sector_agg = df.groupby(['Date', 'Sector'])['Cumulative Change'].mean().reset_index()

# Calculate last cumulative change for each stock for sorting
last_stock_changes = df.groupby('Ticker').apply(
    lambda x: x.dropna(subset=['Cumulative Change']).iloc[-1]['Cumulative Change'] if not x.dropna(subset=['Cumulative Change']).empty else -float('inf'),
    include_groups=False
).to_dict()

# Calculate last cumulative change for each sector for sorting
last_sector_changes = df_sector_agg.groupby('Sector').apply(
    lambda x: x.dropna(subset=['Cumulative Change']).iloc[-1]['Cumulative Change'] if not x.dropna(subset=['Cumulative Change']).empty else -float('inf'),
    include_groups=False
).to_dict()

# 3. Setup Figure and Global Variables
fig = go.Figure()

# Colors for individual stocks (reused for sectors)
colors = px.colors.qualitative.Plotly

# Map sectors to colors for consistency and sort available_sectors by last cumulative change
available_sectors = sorted(df['Sector'].dropna().unique().tolist(), key=lambda s: last_sector_changes.get(s, -float('inf')), reverse=True)
sector_colors = {sector: colors[i % len(colors)] for i, sector in enumerate(available_sectors)}

individual_annotations = []
sector_annotations = []

# 4. Build Individual Stock Traces and Badge Annotations
# These are initially visible
individual_trace_start_idx = len(fig.data)
tickers = sorted(df['Ticker'].unique(), key=lambda t: last_stock_changes.get(t, -float('inf')), reverse=True)
for i, ticker in enumerate(tickers):
    df_ticker = df[df['Ticker'] == ticker]
    is_top_five = df_ticker['Top Five?'].iloc[0]
    stock_name = df_ticker['Name'].iloc[0]
    sector = df_ticker['Sector'].iloc[0]
    color = colors[i % len(colors)] # Use a unique color per stock for individual view

    valid_data = df_ticker.dropna(subset=['Cumulative Change'])
    if valid_data.empty: continue
    last_valid_row = valid_data.iloc[-1]

    fig.add_trace(go.Scatter(
        x=df_ticker['Date'], y=df_ticker['Cumulative Change'],
        mode='lines',
        name=ticker,
        line=dict(color=color, width=2),
        meta={'name': stock_name, 'is_top_five': is_top_five, 'sector': sector, 'type': 'individual'},
        visible=True, # Initially visible
        hovertemplate=f"<b>{stock_name} ({ticker})</b><br>Sector: {sector}<br>Change: %{{y:.2%}}<extra></extra>"
    ))
    # Marker Anchor
    fig.add_trace(go.Scatter(
        x=[last_valid_row['Date']], y=[last_valid_row['Cumulative Change']],
        mode='markers', marker=dict(color=color, size=4),
        showlegend=False, visible=True, hoverinfo='skip',
        meta={'type': 'individual_marker', 'is_top_five': is_top_five, 'sector': sector}
    ))

    individual_annotations.append(dict(
        x=last_valid_row['Date'], y=last_valid_row['Cumulative Change'],
        text=f"<b>{last_valid_row['Cumulative Change']:.2%}</b>",
        showarrow=False, xanchor="left", xshift=10,
        font=dict(color="white", size=10, family="Arial Black"),
        bgcolor=color, bordercolor=color, borderwidth=2, borderpad=4,
        visible=True # Initially visible
    ))

# 5. Build Sector Aggregated Traces and Badge Annotations
# These are initially hidden
sector_trace_start_idx = len(fig.data)
for i, sector_name in enumerate(available_sectors):
    df_sector = df_sector_agg[df_sector_agg['Sector'] == sector_name]
    color = sector_colors[sector_name]

    valid_data = df_sector.dropna(subset=['Cumulative Change'])
    if valid_data.empty: continue
    last_valid_row = valid_data.iloc[-1]

    fig.add_trace(go.Scatter(
        x=df_sector['Date'], y=df_sector['Cumulative Change'],
        mode='lines',
        name=sector_name,
        line=dict(color=color, width=2),
        meta={'sector': sector_name, 'type': 'sector_aggregated'},
        visible=False, # Initially hidden
        hovertemplate=f"<b>{sector_name}</b><br>Change: %{{y:.2%}}<extra></extra>"
    ))
    # Marker Anchor
    fig.add_trace(go.Scatter(
        x=[last_valid_row['Date']], y=[last_valid_row['Cumulative Change']],
        mode='markers', marker=dict(color=color, size=4),
        showlegend=False, visible=False, hoverinfo='skip',
        meta={'type': 'sector_aggregated_marker', 'sector': sector_name}
    ))

    sector_annotations.append(dict(
        x=last_valid_row['Date'], y=last_valid_row['Cumulative Change'], # Corrected y to be a scalar, not a list
        text=f"<b>{last_valid_row['Cumulative Change']:.2%}</b>",
        showarrow=False, xanchor="left", xshift=10,
        font=dict(color="white", size=10, family="Arial Black"),
        bgcolor=color, bordercolor=color, borderwidth=2, borderpad=4,
        visible=False # Initially hidden
    ))

all_annotations = individual_annotations + sector_annotations

# 6. Slicer Logic - Filtering based on Display Mode, Company Name, Top 5, and Sector
def get_args(display_mode='individual', name_filter=None, top_5_only=False, sector_filter=None):
    trace_vis = [False] * len(fig.data)
    ann_vis = [False] * len(all_annotations)

    for i, trace in enumerate(fig.data):
        meta = trace.meta if trace.meta else {}
        trace_type = meta.get('type')

        is_trace_visible = False
        if display_mode == 'individual' and trace_type and 'individual' in trace_type:
            is_trace_visible = True
            if name_filter and meta.get('name') != name_filter: is_trace_visible = False
            if top_5_only and meta.get('is_top_five') != 'Yes': is_trace_visible = False
            if sector_filter and meta.get('sector') != sector_filter: is_trace_visible = False
        elif display_mode == 'sector_aggregated' and trace_type and 'sector_aggregated' in trace_type:
            is_trace_visible = True
            if sector_filter and meta.get('sector') != sector_filter: is_trace_visible = False

        trace_vis[i] = is_trace_visible

    # Handle annotations visibility
    for i, ann_dict in enumerate(all_annotations):
        is_ann_visible = False
        if i < len(individual_annotations): # This is an individual stock annotation
            current_trace_meta = fig.data[individual_trace_start_idx + 2*i].meta # Meta of the corresponding line trace

            if display_mode == 'individual':
                # Individual annotations are visible if no filters are active (show all)
                # OR if they match ALL active filters.
                if not name_filter and not top_5_only and not sector_filter:
                    is_ann_visible = True
                else:
                    matches_name = (name_filter is None) or (current_trace_meta.get('name') == name_filter)
                    matches_top_5 = (not top_5_only) or (current_trace_meta.get('is_top_five') == 'Yes')
                    matches_sector = (sector_filter is None) or (current_trace_meta.get('sector') == sector_filter)

                    if matches_name and matches_top_5 and matches_sector:
                        is_ann_visible = True

        else: # This is a sector aggregated annotation
            j = i - len(individual_annotations)
            current_trace_meta = fig.data[sector_trace_start_idx + 2*j].meta # Meta of the corresponding line trace

            if display_mode == 'sector_aggregated':
                # Sector annotations are visible if no sector filter is active (show all)
                # OR if they match the active sector filter.
                if not sector_filter:
                    is_ann_visible = True
                else:
                    if current_trace_meta.get('sector') == sector_filter:
                        is_ann_visible = True

        ann_vis[i] = is_ann_visible

    updated_anns = [dict(ann, visible=vis) for ann, vis in zip(all_annotations, ann_vis)]
    return [{"visible": trace_vis}, {"annotations": updated_anns}]


# 7. Build Dropdown Options using Company Names and Sectors
# Company name dropdown - always for individual view
name_to_ticker_map = df_summary.set_index('Name')['Ticker'].to_dict()
available_names = sorted(df['Name'].unique().tolist(),
                                       key=lambda n: last_stock_changes.get(name_to_ticker_map.get(n, ''), -float('inf')),
                                       reverse=True)
name_dropdown = [dict(label="All Companies", method="update", args=get_args(display_mode='individual'))]
for name in available_names:
    name_dropdown.append(dict(label=name, method="update", args=get_args(display_mode='individual', name_filter=name)))

# Sector dropdown for individual view
sector_dropdown_individual = [dict(label="All Sectors", method="update", args=get_args(display_mode='individual'))]
for sector in available_sectors:
    sector_dropdown_individual.append(dict(label=sector, method="update", args=get_args(display_mode='individual', sector_filter=sector)))

# Sector dropdown for aggregated view
sector_dropdown_aggregated = [dict(label="All Sectors", method="update", args=get_args(display_mode='sector_aggregated'))]
for sector in available_sectors:
    sector_dropdown_aggregated.append(dict(label=sector, method="update", args=get_args(display_mode='sector_aggregated', sector_filter=sector)))


# 8. Layout and Formatting
fig.update_layout(
    updatemenus=[
        # Display Mode Buttons (index 0)
        dict(type="buttons", direction="left", buttons=[
                # When Individual View is selected
                dict(label="Individual View", method="update", args=[
                    {'visible': get_args(display_mode='individual')[0]['visible']},
                    {'annotations': get_args(display_mode='individual')[1]['annotations'],
                     'updatemenus[3].visible': True, 'updatemenus[4].visible': False}
                ]),
                # When Aggregated by Sector is selected
                dict(label="Aggregated by Sector", method="update", args=[
                    {'visible': get_args(display_mode='sector_aggregated')[0]['visible']},
                    {'annotations': get_args(display_mode='sector_aggregated')[1]['annotations'],
                     'updatemenus[3].visible': False, 'updatemenus[4].visible': True}
                ])
            ], x=0.0, y=1.22, xanchor="left", yanchor="top",
            font=dict(size=10, color="gray"),
            bordercolor="lightgray", borderwidth=1,
            bgcolor="lightblue"
        ),
        # Top 5 / Show All Buttons (index 1) - for Individual View only
        dict(type="buttons", direction="left", buttons=[
                dict(label="Show All", method="update", args=get_args(display_mode='individual')),
                dict(label="Top 5 Only", method="update", args=get_args(display_mode='individual', top_5_only=True))
            ], x=0.25, y=1.22, xanchor="left", yanchor="top",
            font=dict(size=10, color="gray"),
            bordercolor="lightgray", borderwidth=1,
            bgcolor="lightblue"
        ),
        # Company Name Dropdown (index 2) - for Individual View only
        dict(type="dropdown", direction="down", buttons=name_dropdown, x=0.5, y=1.22, xanchor="left", yanchor="top",
            font=dict(size=10, color="gray"),
            bordercolor="lightgray", borderwidth=1,
            bgcolor="lightblue"
        ),
        # Sector Dropdown (for Individual View) (index 3) - initially visible
        dict(type="dropdown", direction="down", buttons=sector_dropdown_individual, x=0.75, y=1.22, xanchor="left", yanchor="top",
            font=dict(size=10, color="gray"),
            bordercolor="lightgray", borderwidth=1,
            bgcolor="lightblue",
            visible=True
        ),
        # Sector Dropdown (for Aggregated View) (index 4) - initially hidden
        dict(type="dropdown", direction="down", buttons=sector_dropdown_aggregated, x=0.75, y=1.22, xanchor="left", yanchor="top",
            font=dict(size=10, color="gray"),
            bordercolor="lightgray", borderwidth=1,
            bgcolor="lightblue",
            visible=False
        )
    ],
    xaxis=dict(
        title="Date",
        range=[min_date, max_date],
        type='date',
        tickformat="%b %Y",
        dtick="M1",
        tickangle=-45
    ),
    yaxis=dict(title="Cumulative Change", tickformat=".0%"),
    template="plotly_white", height=750,
    margin=dict(r=300, t=100),
    legend=dict(orientation="v", x=1.05, y=1, xanchor="left"),
    hovermode="x unified",
    annotations=all_annotations
)

fig.show()

In [5]:
fig.write_html("cumulative_change_chart.html", include_plotlyjs=True)